In [10]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:llama-3.3-70b-versatile")

In [ ]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title:str = Field(description="Title of the movie")
    year:int = Field(description="Year of the movie")
    director:str = Field(description="Director of the movie")
    rating:float = Field(description="Rating of the movie out of 10")


In [ ]:
 model_with_struc = model.with_structured_output(Movie)
 model_with_struc.invoke("Provide details of spider man")

MESSAGE OUTPUT ALONGSIDE PARSED STRUCTURE

In [ ]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    """A Movie with detail"""
    title:str = Field(...,description="Title of the movie")
    year:int = Field(...,description="Year of the movie")
    director:str = Field(...,description="Director of the movie")
    rating:float = Field(...,description="Rating of the movie out of 10")
model_with_struc = model.with_structured_output(Movie,include_raw=True)
model_with_struc.invoke("Provide details of spider man")



NESTED STRUCTURE

In [11]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str
class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float|None = Field(None,description="Budget in millions USD")


model_with_struc = model.with_structured_output(MovieDetails)
model_with_struc.invoke("Provide details of spider man")

MovieDetails(title='Spider-Man', year=2002, cast=[Actor(name='Tobey Maguire', role='Spider-Man'), Actor(name='Willem Dafoe', role='Green Goblin')], genres=['Action', 'Adventure'], budget=139.0)

TYPED DICT

In [12]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8.1, 'title': 'Avengers', 'year': 2012}

In [13]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'Avengers',
 'year': 2012}

In [15]:
model.profile


{'name': 'Llama 3.3 70B Versatile',
 'release_date': '2024-12-06',
 'last_updated': '2024-12-06',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 32768,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

DATA CLASSES

In [16]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='60e8c4b6-8e7e-433e-8b8b-a9c1e51b050a'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'qcbec4egh', 'function': {'arguments': '{"email":"john@example.com","name":"John Doe","phone":"(555) 123-4567"}', 'name': 'ContactInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 288, 'total_tokens': 321, 'completion_time': 0.060930607, 'completion_tokens_details': None, 'prompt_time': 0.01509185, 'prompt_tokens_details': None, 'queue_time': 0.008384598, 'total_time': 0.076022457}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a00554-4f7a-7620-a42f-a564d9d11c36-0', tool_calls=[{'name': 'ContactInfo', 'args': {'email': 'jo

In [17]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')